In [2]:
!python extract_features_fp.py --data_h5_dir "E:/BS/CLAM/RESULTS_DIRECTORY" --data_slide_dir "E:/BS/CLAM/DATA_DIRECTORY" --csv_path "E:/BS/CLAM/RESULTS_DIRECTORY/process_list_autogen.csv" --feat_dir "E:/BS/CLAM/FEATURES_DIRECTORY" --batch_size 256 --slide_ext .tif


initializing dataset
loading model checkpoint
TimmCNNEncoder(
  (model): FeatureListNet(
    (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
    (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (act1): ReLU(inplace=True)
    (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
    (layer1): Sequential(
      (0): Bottleneck(
        (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (act1): ReLU(inplace=True)
        (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (drop_block): Identity()
        (act2): ReLU(inplace=True)
        (aa): Identity()
        (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), 


  0%|          | 0/123 [00:00<?, ?it/s]

  0%|          | 0/3 [00:00<?, ?it/s]

 33%|███▎      | 1/3 [00:31<01:02, 31.34s/it]

100%|██████████| 3/3 [00:32<00:00, 10.93s/it]

 85%|████████▌ | 105/123 [00:32<00:05,  3.20it/s]

  0%|          | 0/2 [00:00<?, ?it/s]

 50%|█████     | 1/2 [00:30<00:30, 30.97s/it]

100%|██████████| 2/2 [00:32<00:00, 16.10s/it]

 86%|████████▌ | 106/123 [01:05<00:12,  1.35it/s]

  0%|          | 0/1 [00:00<?, ?it/s]

100%|██████████| 1/1 [00:30<00:00, 30.70s/it]

 87%|████████▋ | 107/123 [01:35<00:20,  1.31s/it]

  0%|          | 0/3 [00:00<?, ?it/s]

 33%|███▎      | 1/3 [00:31<01:03, 31.58s/it]

 67%|██████▋   | 2/3 [00:32<00:13, 13.29s/it]

100%|██████████| 3/3 [00:33<00:00, 11.09s/it]

 88%|████████▊ | 108/123 [02:09<00:32,  2.14s/it]

  0%|          | 0/2 [00:00<?, ?it/s]

 50%|█████     | 1/2 [00:31<00:31, 31.30s/it]

100%|██████████| 2/2 [00:32<00:00, 16.26s/it]

 89%|████████▊ | 109/123 [02:41<00:45,  3.24s/it]

  0%|          | 0/2 [00:00<?, ?it/s]


In [5]:
import os
import random
import shutil
import csv

# ------------------- 配置参数 -------------------
pt_dir = r'E:\BS\CLAM\FEATURES_DIRECTORY\pt_files'
h5_dir = r'E:\BS\CLAM\FEATURES_DIRECTORY\h5_files'
output_dir = r'E:\BS\CLAM\DATA_ROOT_DIR'
split_ratio = (0.7, 0.15, 0.15)
# -----------------------------------------------

os.makedirs(output_dir, exist_ok=True)
splits = ['train', 'val', 'test']

# 获取样本名列表（无扩展名）
def get_sample_names(dir_path, prefix):
    return sorted([
        os.path.splitext(f)[0]
        for f in os.listdir(dir_path)
        if f.startswith(prefix) and f.endswith('.pt')
    ])

positive_samples = get_sample_names(pt_dir, 'SLTDE_EN')
negative_samples = get_sample_names(pt_dir, 'SLTDE_NO')

# 检查对应 .h5 文件是否存在
def check_h5_files(sample_list):
    return [s for s in sample_list if os.path.exists(os.path.join(h5_dir, s + '.h5'))]

positive_samples = check_h5_files(positive_samples)
negative_samples = check_h5_files(negative_samples)

# 随机划分
def split_data(samples):
    random.shuffle(samples)
    total = len(samples)
    n_train = int(total * split_ratio[0])
    n_val = int(total * split_ratio[1])
    return (
        samples[:n_train],
        samples[n_train:n_train + n_val],
        samples[n_train + n_val:]
    )

pos_train, pos_val, pos_test = split_data(positive_samples)
neg_train, neg_val, neg_test = split_data(negative_samples)

# 合并带标签
def label_and_tag(samples, label, split):
    return [{'name': s, 'label': label, 'split': split} for s in samples]

all_data = (
    label_and_tag(pos_train, 1, 'train') +
    label_and_tag(pos_val, 1, 'val') +
    label_and_tag(pos_test, 1, 'test') +
    label_and_tag(neg_train, 0, 'train') +
    label_and_tag(neg_val, 0, 'val') +
    label_and_tag(neg_test, 0, 'test')
)

# 初始化目标子目录
for split in splits:
    os.makedirs(os.path.join(output_dir, split, 'pt'), exist_ok=True)
    os.makedirs(os.path.join(output_dir, split, 'h5'), exist_ok=True)

# 拷贝文件并记录各 split 的 CSV 数据
split_csv_data = {split: [] for split in splits}

for item in all_data:
    name = item['name']
    label = item['label']
    split = item['split']

    pt_src = os.path.join(pt_dir, name + '.pt')
    h5_src = os.path.join(h5_dir, name + '.h5')

    pt_dest = os.path.join(output_dir, split, 'pt', name + '.pt')
    h5_dest = os.path.join(output_dir, split, 'h5', name + '.h5')

    shutil.copy(pt_src, pt_dest)
    shutil.copy(h5_src, h5_dest)

    split_csv_data[split].append({
        'case_id': name,
        'slide_id': name + '.pt',
        'label': label
    })

# 写入各 split 的 CSV 文件
for split in splits:
    csv_path = os.path.join(output_dir, split, 'dataset.csv')
    with open(csv_path, mode='w', newline='') as csvfile:
        writer = csv.DictWriter(csvfile, fieldnames=['case_id', 'slide_id', 'label'])
        writer.writeheader()
        writer.writerows(split_csv_data[split])

print("✅ 所有数据处理完成，CSV 文件分别写入各子目录。")


✅ 所有数据处理完成，CSV 文件分别写入各子目录。


In [3]:
!python main.py --task task_1_tumor_vs_normal --data_root_dir features --results_dir results/endometriosis --max_epochs 20 --batch_size 1 --early_stopping --split_dir "E:/BS/CLAM/splits/task_1_tumor_vs_normal_100"





Load Dataset
label column: label
label dictionary: {0: 0, 1: 1}
number of classes: 2
slide-level counts:  
 label
1    93
0    30
Name: count, dtype: int64
Patient-LVL; Number of samples registered in class 0: 30
Slide-LVL; Number of samples registered in class 0: 30
Patient-LVL; Number of samples registered in class 1: 93
Slide-LVL; Number of samples registered in class 1: 93
split_dir:  E:/BS/CLAM/splits/task_1_tumor_vs_normal_100
################# Settings ###################
num_splits:  10
k_start:  -1
k_end:  -1
task:  task_1_tumor_vs_normal
max_epochs:  20
results_dir:  results/endometriosis
lr:  0.0001
experiment:  None
reg:  1e-05
label_frac:  1.0
bag_loss:  ce
seed:  1
model_type:  clam_sb
model_size:  small
use_drop_out:  0.25
weighted_sample:  False
opt:  adam
bag_weight:  0.7
inst_loss:  None
B:  8
split_dir:  E:/BS/CLAM/splits/task_1_tumor_vs_normal_100

Load Dataset
label column: label
label dictionary: {0: 0, 1: 1}
number of classes: 2
slide-level counts:  
 label
1   

In [13]:
import pandas as pd
import json
import os

# 修改为你自己的路径
base_dir = r'E:\BS\CLAM\DATA_ROOT_DIR'
split_save_dir = r'E:\BS\CLAM\splits\task_1_tumor_vs_normal_100'
os.makedirs(split_save_dir, exist_ok=True)

# 读取csv
train_df = pd.read_csv(os.path.join(base_dir, r'E:\BS\CLAM\DATA_ROOT_DIR\DATASET_1_DATA_DIR\dataset.csv'))
val_df = pd.read_csv(os.path.join(base_dir, r'E:\BS\CLAM\DATA_ROOT_DIR\DATASET_2_DATA_DIR\dataset.csv'))
test_df = pd.read_csv(os.path.join(base_dir, r'E:\BS\CLAM\DATA_ROOT_DIR\DATASET_3_DATA_DIR\dataset.csv'))

# 提取slide_id列表
split_dict = {
    'train': train_df['slide_id'].tolist(),
    'val': val_df['slide_id'].tolist(),
    'test': test_df['slide_id'].tolist()
}

# 保存为 splits_0.json
with open(os.path.join(split_save_dir, 'splits_0.json'), 'w') as f:
    json.dump(split_dict, f, indent=4)

print("✅ splits_0.json 已生成完毕！")


✅ splits_0.json 已生成完毕！


In [15]:
import os
import random
import shutil
import csv

# 路径配置
pt_dir = r'E:\BS\CLAM\FEATURES_DIRECTORY\pt_files'
h5_dir = r'E:\BS\CLAM\FEATURES_DIRECTORY\h5_files'
output_dir = r'E:\BS\CLAM\test'
split_ratio = (0.7, 0.15, 0.15)

os.makedirs(output_dir, exist_ok=True)

# 识别样本名（不含扩展名）
def get_sample_names(dir_path, prefix):
    return sorted([
        os.path.splitext(f)[0]
        for f in os.listdir(dir_path)
        if f.startswith(prefix) and f.endswith('.pt')
    ])

positive_samples = get_sample_names(pt_dir, 'SLTDE_EN')
negative_samples = get_sample_names(pt_dir, 'SLTDE_NO')

# 检查 h5 配对存在
def check_h5_files(sample_list):
    valid_samples = []
    for name in sample_list:
        h5_path = os.path.join(h5_dir, name + '.h5')
        if os.path.exists(h5_path):
            valid_samples.append(name)
        else:
            print(f"⚠️ 缺少对应的 .h5 文件: {name}.h5")
    return valid_samples

positive_samples = check_h5_files(positive_samples)
negative_samples = check_h5_files(negative_samples)

# 随机划分数据集
def split_data(samples):
    random.shuffle(samples)
    total = len(samples)
    n_train = int(total * split_ratio[0])
    n_val = int(total * split_ratio[1])
    return (
        samples[:n_train],
        samples[n_train:n_train + n_val],
        samples[n_train + n_val:]
    )

pos_train, pos_val, pos_test = split_data(positive_samples)
neg_train, neg_val, neg_test = split_data(negative_samples)

# 合并并带标签
def label_and_tag(samples, label, split):
    return [{'name': s, 'label': label, 'split': split} for s in samples]

all_data = (
    label_and_tag(pos_train, 1, 'train') +
    label_and_tag(pos_val, 1, 'val') +
    label_and_tag(pos_test, 1, 'test') +
    label_and_tag(neg_train, 0, 'train') +
    label_and_tag(neg_val, 0, 'val') +
    label_and_tag(neg_test, 0, 'test')
)

# 创建目标文件夹
for split in ['train', 'val', 'test']:
    os.makedirs(os.path.join(output_dir, split), exist_ok=True)

# 复制文件并记录
csv_data = []

for item in all_data:
    name = item['name']
    label = item['label']
    split = item['split']
    
    pt_src = os.path.join(pt_dir, name + '.pt')
    h5_src = os.path.join(h5_dir, name + '.h5')
    
    pt_dest = os.path.join(output_dir, split, name + '.pt')
    h5_dest = os.path.join(output_dir, split, name + '.h5')
    
    shutil.copy(pt_src, pt_dest)
    shutil.copy(h5_src, h5_dest)
    
    csv_data.append({
        'filename_pt': os.path.join(split, name + '.pt'),
        'filename_h5': os.path.join(split, name + '.h5'),
        'label': label,
        'split': split
    })

# 写入 CSV
csv_path = os.path.join(output_dir, 'dataset.csv')
with open(csv_path, mode='w', newline='') as csvfile:
    writer = csv.DictWriter(csvfile, fieldnames=['filename_pt', 'filename_h5', 'label', 'split'])
    writer.writeheader()
    writer.writerows(csv_data)

print("✅ 数据集拆分完成，CSV 文件生成于:", csv_path)


✅ 数据集拆分完成，CSV 文件生成于: E:\BS\CLAM\test\dataset.csv


In [32]:
import pandas as pd
from sklearn.model_selection import StratifiedKFold
import os

# 读取数据
csv_path = r"E:\BS\CLAM\slide_labels.csv"
df = pd.read_csv(csv_path)

# 检查字段
df.columns = df.columns.str.strip()
assert {'case_id', 'slide_id', 'label'}.issubset(df.columns)

# 设置输出路径
split_save_dir = r"E:\BS\CLAM\splits\task_1_tumor_vs_normal_100"
os.makedirs(split_save_dir, exist_ok=True)

# 十折交叉验证
skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
X = df['case_id']
y = df['label']

for fold, (train_val_idx, test_idx) in enumerate(skf.split(X, y)):
    train_val_df = df.iloc[train_val_idx]
    test_df = df.iloc[test_idx]

    # 再从train_val中划分val
    skf_inner = StratifiedKFold(n_splits=9, shuffle=True, random_state=fold)
    train_idx, val_idx = next(skf_inner.split(train_val_df['case_id'], train_val_df['label']))
    train_df = train_val_df.iloc[train_idx]
    val_df = train_val_df.iloc[val_idx]

    # 保存 splits_x.csv
    split_df = pd.DataFrame({
        'train': train_df['case_id'].reset_index(drop=True),
        'val': val_df['case_id'].reset_index(drop=True),
        'test': test_df['case_id'].reset_index(drop=True)
    })
    split_df.to_csv(os.path.join(split_save_dir, f'splits_{fold}.csv'), index=False)

    # 保存 splits_x_bool.csv
    bool_df = pd.DataFrame(index=df['case_id'].unique())
    bool_df['train'] = bool_df.index.isin(train_df['case_id'])
    bool_df['val'] = bool_df.index.isin(val_df['case_id'])
    bool_df['test'] = bool_df.index.isin(test_df['case_id'])
    bool_df = bool_df.astype(str).apply(lambda x: x.str.upper())  # 转为 TRUE/FALSE
    bool_df.to_csv(os.path.join(split_save_dir, f'splits_{fold}_bool.csv'))

    # 保存 splits_x_descriptor.csv
    def count_labels(subset):
        return subset['label'].value_counts().sort_index().tolist()

    labels = sorted(df['label'].unique())
    desc_df = pd.DataFrame(index=[f'subtype_{i+1}' for i in labels])
    desc_df['train'] = train_df['label'].value_counts().reindex(labels).fillna(0).astype(int).values
    desc_df['val'] = val_df['label'].value_counts().reindex(labels).fillna(0).astype(int).values
    desc_df['test'] = test_df['label'].value_counts().reindex(labels).fillna(0).astype(int).values
    desc_df.to_csv(os.path.join(split_save_dir, f'splits_{fold}_descriptor.csv'))

split_save_dir


'E:\\BS\\CLAM\\splits\\task_1_tumor_vs_normal_100'

In [31]:
import os
import csv

# 文件夹路径
folder_path = r'E:\BS\CLAM\FEATURES_DIRECTORY\pt_files'

# 输出的 csv 文件名
output_csv = 'slide_labels.csv'

# 收集数据
data = []

for filename in os.listdir(folder_path):
    if filename.endswith('.pt') or filename.endswith('.tif') or filename.endswith('.ndpi'):  # 根据实际扩展名调整
        name_no_ext = os.path.splitext(filename)[0]
        if 'EN' in name_no_ext.upper():
            label = 1
        elif 'NO' in name_no_ext.upper():
            label = 0
        else:
            continue  # 忽略不符合命名规则的文件
        data.append([name_no_ext, name_no_ext, label])

# 写入 csv 文件
with open(output_csv, 'w', newline='') as f:
    writer = csv.writer(f)
    writer.writerow(['case_id', 'slide_id', 'label'])
    writer.writerows(data)

print(f"{output_csv} 已生成，包含 {len(data)} 条记录。")


slide_labels.csv 已生成，包含 123 条记录。
